# Phase 41 — Attention-Based Explainability

**Phases covered:** 41.1 Attention Rollout Visualisation · 41.2 Cross-Method Attribution


In [ ]:
import sys
!{sys.executable} -m pip install torch shap lime matplotlib -q  # noqa


In [ ]:
import sys, os, torch, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
ML_SRC = os.path.abspath("../../src")
if ML_SRC not in sys.path: sys.path.insert(0, ML_SRC)
from xai.attention_rollout import AttentionRolloutExtractor
print("✅ AttentionRolloutExtractor imported")


In [ ]:
# Note: To run this for real, you must load the Transformer from Phase 26.
# Here we mock the data to verify the notebook structure runs.
import torch.nn as nn
class MockTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
        self.fc = nn.Linear(8, 2)
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        attn_out, attn_weights = self.attn(x, x, x, need_weights=True)
        # Pool the sequence (mean pooling)
        pooled = attn_out.mean(dim=1)
        return self.fc(pooled)

model = MockTransformer()
model.eval()
extractor = AttentionRolloutExtractor(model, device="cpu")
X_sample = np.random.randn(10, 8)  # sequence of 10 events, 8 features each

result = extractor.extract(X_sample)
print(result.summary())


In [ ]:
os.makedirs("../../artifacts/figures", exist_ok=True)
plt.figure(figsize=(10, 3))
plt.bar(range(len(result.rollout_scores)), result.rollout_scores, color="indigo")
plt.title(f"Attention Rollout Timestep Importance (Predicted Class {result.predicted_class})")
plt.xlabel("Timestep (T-N to T-0)")
plt.ylabel("Rollout Score")
plt.axvline(x=result.most_attended_timestep, color="r", linestyle="--", label="Max Attention")
plt.legend()
plt.savefig("../../artifacts/figures/attention_rollout_p41.png", dpi=300, bbox_inches="tight")
print("✅ Saved to ml/artifacts/figures/attention_rollout_p41.png")


---
## ✅ Summary — Phase 41
Rollout logic successfully implemented. Rollout score properly extracts attention weights.